In [ ]:
from __future__ import annotations
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from pathlib import Path
import dataretrieval.nwis as nwis
import dataretrieval.waterdata as waterdata
import datetime
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynhd as nhd
from pynhd import NLDI, NHDPlusHR, WaterData
import py3dep
import pygeohydro as gh
from pathlib import Path
import networkx as nx
import xarray as xr
import xrspatial
import os
from scripts import maps, data, dataprocessing, SNOTEL_Analyzer, files, catchment, stream_analysis


In [ ]:
nldi = NLDI()
station = '09114500' #GUNNISON RIVER above blue mesa reservoir
WY = 2025 #2025 water year ends on oct 1 2025, so the DOI falles within this water year

basin = nldi.get_basins(station)
basinname = 'BlueMesaReservoir'
geometry = basin.geometry.iloc[0]

## Make data files

In [ ]:
#Basin File
site_feature, upstream_network = files.basinFile(basin, basinname, station)

#SNOTEL File
snotel = files.snotelFile(geometry,'CO')

#Streamflow File
discharge = files.dischargeFile(station)

# 1. Area of Interest

### Watershed Map with stream gauge and SNOTEL icons 

In [ ]:
maps.snotel_mapping(snotel, basin, site_feature)

### Catchment Characteristics

In [ ]:
#Get information for Figures/future Dataframes

#Flowlines for the mainstem of the station
flw_main = catchment.nldi_info(station,"upstreamMain",'flowlines')

#Flowlines for the Tributaries of the station
flw_trib = catchment.nldi_info(station,'upstreamTributaries','flowlines')

#Get the stations for the upstream of the station within 1000 km
st_all = catchment.nldi_info(station,'upstreamTributaries','nwissite')

#List of active station ids 
st_active = catchment.active(st_all)

#Flow points where HUC12 boundaries intersect with the flowlines for the tributaries of the station
pp = catchment.nldi_info(station,'upstreamTributaries','huc12pp')


In [ ]:
#Slope
slope = catchment.slope(flw_trib)

#Plot the slope attribute for the flowlines of the tributaries of the station
ax = basin.plot(facecolor="none", edgecolor="k", figsize=(8, 6))
slope.plot(
    ax=ax,  
    column="slope",
    cmap="plasma",
    legend=True,
    legend_kwds={"label": "Slope (m/m)"},
)

st_active.plot(
    ax=ax,
    label="USGS stations",
    marker="v",
    markersize=100,
    zorder=5,
    color="darkblue",
)
pp.plot(ax=ax, label="HUC12 pour points", marker="o", markersize=50, color="k", zorder=3)

ax.legend(loc="best")
ax.set_aspect("auto")
ax.set_axis_off()
ax.figure.set_dpi(100)

ax.set_axis_off()

fig = ax.get_figure()
fig.savefig("figures/slope_attrib.png", bbox_inches="tight", facecolor="w")

In [ ]:
#Topography

topo = catchment.DEM(geometry,station)

fig, axs = plt.subplots(ncols=2, figsize=(13, 4))
topo['elevation'].plot(ax=axs[0])
topo['slope'].plot(ax=axs[1])
for ax in axs:
    ax.set_title("")
    ax.set_axis_off()
fig.savefig("Figures/dem_slope.png", bbox_inches="tight", facecolor="w")

### Land Cover

In [ ]:
#Initial basin dataframe (elevation,slope,area)
temp_info = catchment.basin_dataframe(topo,geometry,basinname,basin,station)

In [ ]:
#List of data years after I had an error, so most recent info is 2021
# cover: 2021, 2019, 2016, 2013, 2011, 2008, 2006, 2004, 2001
# canopy: 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021
# descriptor: 2021, 2019, 2016, 2013, 2011, 2008, 2006, 2004, 2001

descript = gh.nlcd_bygeom(basin.geometry, 100, years={"descriptor": 2021}, ssl=False)
lulcover = gh.nlcd_bygeom(basin.geometry, 100, years={"cover": [2019, 2021]}, ssl=False)

stats = gh.cover_statistics(lulcover[f"{station}"].cover_2021) # Get the land cover statistics 
roughness = gh.overland_roughness(lulcover[f"{station}"].cover_2021) # Get the overland flow roughness


In [ ]:
#Create map for Land Cover
cmap, norm, levels = gh.plot.cover_legends()
cover = lulcover[f"{station}"].cover_2021 
unique = np.unique(cover.values)

nlcd_legend = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land (Rock/Sand/Clay)",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
    127: "No Data / Masked"
}

unique = [nlcd_legend.get(code, "Unknown") for code in unique]

In [ ]:
def map_names(val):
    return nlcd_legend.get(val, "Unknown")

v_map_names = np.vectorize(map_names) 
cover_names = xr.apply_ufunc(v_map_names, cover)
vals, counts = np.unique(cover.values, return_counts=True)

# Summary DataFrame
summary = pd.DataFrame({
    "code": vals,
    "count": counts,
    "name": [nlcd_legend.get(v, "Unknown") for v in vals]
})

# Calculate Percentage
total_pixels = summary[summary['code'] != 127]['count'].sum()
summary['percent'] = (summary['count'] / total_pixels) * 100
summary = summary[summary['code'] != 127]

lulcover = summary.copy()
lulcover = lulcover.drop(columns=["count", "code"])

lulcover.rename(columns={"name": "Basin_Name", 'percent': temp_info['Basin_Name'].values[0]}, inplace=True)
lulcover.set_index("Basin_Name", inplace=True)

lulcover = lulcover.T
lulcover.columns = lulcover.columns.str.replace(" ", "_").str.replace("(", "").str.replace(")", "").str.replace("/", "_")

#Combine into final summary
temp_info.set_index('Basin_Name', inplace=True)
basin_info = pd.concat([temp_info, lulcover], axis=1)

OutputFolder = 'files/basin_info'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)
basin_info.to_csv(f'{OutputFolder}/basin_info_{station}.csv')

In [ ]:
# Mask 127 to make it transparent
cover_masked = cover.where(cover != 127)

fig, ax = plt.subplots(figsize=(10, 8))
im = cover_masked.plot(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)

plot_values = [v for v in unique if v != 127]
plot_names = [nlcd_legend.get(v, "Unknown") for v in plot_values]

cbar = fig.colorbar(im, ax=ax, ticks=plot_values, fraction=0.046, pad=0.04)
cbar.ax.set_yticklabels(plot_names)
ax.set_title(f"Land Cover: {station} (2021)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
cover.where(cover < 127).plot(ax=ax1, cmap=cmap, levels=levels, cbar_kwargs={"ticks": levels[:-1]})
ax1.set_title("Land Use/Land Cover 2021")
ax1.set_axis_off()

roughness.plot(ax=ax2)
ax2.set_title("Overland Roughness")
ax2.set_axis_off()
fig.savefig(Path("Figures", "lulc.png"), bbox_inches="tight", facecolor="w")

### Catchment Summary 

In [ ]:
basin_info

# 2. SWE Analysis for Snotel 

### Sites descriptions 

In [ ]:
snotel

### Historical Comparison

In [ ]:

# Load the data for one site
sites = list(snotel.code)
print(sites)
stateab = 'Co'
sitedict = dict()

for site in sites:
    sitedict[site] = dataprocessing.processSNOTEL(site, stateab, WY)

In [ ]:
watershed = "Upper Gunnison"
AOI = 'Above Blue Mesa Reservoir'
DOI = '04-01'
SNOTEL_Analyzer.SNOTELPlots(sitedict, snotel, WY, watershed, AOI,DOI)

#Note that Park Cone stopped measuring SWE after 11-27 of 2025, only has 58 data points. 
# This is true for most WY at this site 

In [ ]:
catchmentswe = SNOTEL_Analyzer.catchmentSNOTELAnalysis(sitedict, WY, watershed, AOI, DOI)

# 3. Streamflow Analysis

### Location Information 

In [ ]:
info, met = waterdata.get_monitoring_locations(monitoring_location_id= f"USGS-{station}")
info = info.T.dropna()
info.head(45)

### Historical Comparison for April-September of Streamflow Volume (m^3)

In [ ]:
#Clean up data and include water-year column
streamflow = discharge.drop(columns=['site_no']).copy()
streamflow.rename(columns={'flow_cms':'Flow (cms)'}, inplace=True)
streamflow["Flow Volume (m^3)"] = streamflow['Flow (cms)'] *60*60*24
streamflow = streamflow.drop(columns=['Flow (cms)'])
streamflow.reset_index(inplace=True)

In [ ]:
processed = stream_analysis.process_stream(streamflow, WY)
stream_analysis.StreamPlots(processed, WY, watershed, AOI,DOI)

# 4. Peak Parity with SWE, April-September

In [ ]:
catchmentswe.reset_index(inplace=True)

In [ ]:
catchmentswe['M-D'] = pd.to_datetime(catchmentswe['M-D'],format='%m-%d')
catchmentswe.set_index('M-D', inplace=True)

In [ ]:
monthdict = {1:("April",4),2:("May",5),3:("June",6),4:("July",7),5:("August",8),6:("September",9)}
butte = sitedict['380_CO_SNTL']
park = sitedict['680_CO_SNTL']
taylor = sitedict['1141_CO_SNTL']

title = f'Historical Peak SWE Parity Plots from {watershed} Basin \n {AOI} for April-September'

fig, axs = plt.subplots(2, 3, figsize = (10, 8))
fig.suptitle(title)
opacity = 0.25

axs = axs.ravel()
for i,key in enumerate(monthdict.keys()):
    df1 = butte[butte['M'] == monthdict[key][1]].copy()
    df2 = park[park['M'] == monthdict[key][1]].copy()
    df3 = taylor[taylor['M'] == monthdict[key][1]].copy()
    catchment = catchmentswe[catchmentswe.index.month == monthdict[key][1]].copy()
    
    year_cols1 = [col for col in df1.columns if '_SWE_in' in col]
    year_cols2 = [col for col in df2.columns if '_SWE_in' in col]
    year_cols3 = [col for col in df3.columns if '_SWE_in' in col]
    axs[i].set_title(f"Streamflow Volume for {monthdict[key][0]}")

    for df in [df1, df2, df3]:
        df['date'] = pd.to_datetime({'year': 2023, 'month': df['M'],'day': df['D'] })
        df.set_index(df['date'], inplace=True)

    axs[i].plot(catchment.index,catchment['max'],color='red',label='Peak SWE')

    for col in year_cols1:
        axs[i].scatter(df1.index, df1[col].values, color='green', label='Butte Site', alpha=0.5)
    for col in year_cols2:
        axs[i].scatter(df2.index, df2[col].values, color='blue', label='Park Cone Site', alpha=0.5)
    for col in year_cols3:
        axs[i].scatter(df3.index, df3[col].values, color='orange', label='Upper Taylor Site', alpha=0.5)

    axs[i].xaxis.set_major_locator(ticker.MaxNLocator(4))
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    axs[i].tick_params(labelrotation=45)

    axs[i].set_xlabel('Date')
    axs[i].set_ylabel('SWE (inches)')

handles, labels = axs[0].get_legend_handles_labels()
fig.subplots_adjust( hspace=0.5,wspace=0.5)        
fig.legend(handles, labels,loc='lower center',ncol=8, bbox_to_anchor=(.5, -.05))
plt.show()


In [ ]:
butte = sitedict['380_CO_SNTL']
park = sitedict['680_CO_SNTL']
taylor = sitedict['1141_CO_SNTL']


year_cols = [col for col in butte.columns if '_SWE_in' in col]
year_cols